# DMOSpeech 2: Signal-Selection Validation
## Why the SNR gate? Why drift? Why Learn-Then-Test?

**Thesis:** A Risk-Calibrated Adaptive Handover Method for Teacher-Guided Flow-Matching TTS Inference

The proposed adaptive scheduler (Definition 3.4) uses two measurements: the **prompt SNR** (gate) and the **endpoint drift** $d_i$ (stopping signal), with thresholds calibrated by **Learn-Then-Test**. This notebook tests each choice against alternatives under one frozen protocol:

| Question | Proposal's choice | Compared against |
|---|---|---|
| **Q1.** Which prompt signal should gate early handover? | Energy-percentile prompt SNR | Spectral-percentile SNR, spectral flatness, high-frequency ratio, noise floor, prompt UTMOS, prompt duration, speaking rate, text length; oracle clean/noisy label |
| **Q2.** Which teacher signal should decide when to stop? | Endpoint drift $d_i$ | Cosine change, per-frame max drift, CFG gap, velocity norm; SNR gate + fixed Step 4/5/6; matched-random |
| **Q3.** Which calibration framework? | LTT with exact binomial test | Naive selection, Hoeffding; written comparison with conformal prediction, CRC, RCPS |

**Protocol** (written to `signal_selection_preregistration.json` before any analysis): development (100 speakers) → AUC and signal direction; calibration (200) → threshold by LTT ($\alpha = 0.05$, $\delta = 0.10$); test (200) → held-out robust risk and mean NFE.

**No re-synthesis:** failure labels are reused from `colab/results/gpu_grid_all1000.csv`. The notebook runs the teacher up to Step 6 once per prompt (1,000 prompts) to compute the trajectory signals, and checks that it reproduces the stored $d_4, d_5, d_6$ exactly.

**Estimated runtime:** about 20–40 minutes on a Tesla T4 (resumable if the session disconnects).

---
### Instructions
1. Set the runtime to **GPU** (`Runtime > Change runtime type > T4 GPU`).
2. Click **Runtime > Run all**.
3. The last cell downloads `DMOSpeech2_Signal_Selection_Results.zip` with every output.

### Step 1: Verify GPU Allocation

In [ ]:
!nvidia-smi

### Step 2: Clone Thesis Repository & Install Dependencies

In [ ]:
import os
if not os.path.exists('/content/Shonget-DMOSpeech2'):
    !git clone https://github.com/Shonget-Thesis/Shonget-DMOSpeech2.git /content/Shonget-DMOSpeech2
%cd /content/Shonget-DMOSpeech2
!git checkout main
!git pull origin main

# Same pinned dependencies as the 1,000-utterance GPU retest notebook
!pip install -q "numpy==2.2.3" "pandas==2.2.3"
!pip install -q -r requirements.txt
!pip install -q torchdiffeq vocos speechbrain transformers jiwer soundfile librosa pyarrow scipy jieba accelerate huggingface_hub

import numpy as np, pandas as pd, numba
print(f"Verified: NumPy {np.__version__} | Pandas {pd.__version__} | Numba {numba.__version__} [OK]")
!git log --oneline -1

### Step 3: Download DMOSpeech 2 Checkpoints
`model_85000.pt` (4.0 GB, teacher + student) and `model_1500.pt` (270 MB, duration predictor) into `ckpts/`.

In [ ]:
!mkdir -p ckpts
!wget -c https://huggingface.co/yl4579/DMOSpeech2/resolve/main/model_85000.pt -O ckpts/model_85000.pt
!wget -c https://huggingface.co/yl4579/DMOSpeech2/resolve/main/model_1500.pt -O ckpts/model_1500.pt

### Step 4: Download the Seed-TTS test-en Dataset
Prompt audio for the same 500 speakers (Dev 100 / Calibration 200 / Test 200) used in every earlier test.

In [ ]:
import os

test_file = 'data/seed-tts-eval/en/prompt-wavs/common_voice_en_10933823.wav'

# Check if audio is missing or only a Git LFS pointer
if not os.path.exists(test_file) or os.path.getsize(test_file) < 1000:
    print("Resolving Git LFS binaries for the Seed-TTS evaluation dataset...")
    if os.path.exists('data/seed-tts-eval/.git'):
        !cd data/seed-tts-eval && git lfs install && git lfs pull
    if not os.path.exists(test_file) or os.path.getsize(test_file) < 1000:
        print("Cleaning leftover pointer files and downloading via huggingface_hub...")
        !rm -rf data/seed-tts-eval
        from huggingface_hub import snapshot_download
        snapshot_download(repo_id="zhaochenyang20/seed-tts-eval", repo_type="dataset", local_dir="data/seed-tts-eval")
    print("Dataset setup complete!")
else:
    print("Seed-TTS audio binaries verified!")

!ls -lh data/seed-tts-eval/en/prompt-wavs/common_voice_en_10933823.wav

### Step 5: Smoke Test (10 prompts, ~2 minutes)
Checks models, dataset and feature extraction end to end. Writes to a **separate** folder so it never mixes with the real run.

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
runner = '"Handover Scheduling/colab/run_signal_selection_validation.py"'
out_dir = 'Handover Scheduling/colab/results/signal_selection'

!rm -rf "{out_dir}_smoke"
!python {runner} --device {device} --n_pairs 10 --output_dir "{out_dir}_smoke"

import pandas as pd
smoke = pd.read_csv(f'{out_dir}_smoke/signal_features_all1000.csv')
print(f"Smoke test OK: {len(smoke)} rows, {smoke.shape[1]} columns")
display(smoke[['utt_id', 'condition', 'snr_energy', 'snr_spectral', 'prompt_utmos', 'drift_4', 'cfg_gap_4']].head(10))

### Step 6: Full Run (1,000 prompts) + Analysis
Freezes the protocol, extracts all signals for the 500 speakers × 2 conditions, then runs Q1, Q2 and Q3. **Resumable:** if Colab disconnects, re-run this cell and it continues from the saved CSV.

In [ ]:
!python {runner} --device {device} --output_dir "{out_dir}"

### Step 7: View the Report

In [ ]:
import os, json
from IPython.display import Markdown, display

rep = os.path.join(out_dir, 'SIGNAL_SELECTION_REPORT.md')
if os.path.exists(rep):
    with open(rep, 'r', encoding='utf-8') as f:
        display(Markdown(f.read()))
    s = json.load(open(os.path.join(out_dir, 'signal_selection_summary.json')))
    print("Q1 verdicts:", {k: v['verdict'] for k, v in s['q1_verdict'].items()})
    print("Q2 verdict:", s['q2_verdict']['verdict'])
    print("Reproduction (max |diff| vs stored d_i):", {k: v['max_abs_diff'] for k, v in s['reproduction'].items()})
else:
    print(f"No report yet at {rep}. Run Step 6 first.")

### Step 8: Zip All Outputs and Download
Packs the feature CSV, preregistration, summary JSON, report, the runner script and the repository commit into one ZIP.

In [ ]:
import os, zipfile, subprocess, datetime

zip_name = 'DMOSpeech2_Signal_Selection_Results.zip'
commit = subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
with open(os.path.join(out_dir, 'RUN_INFO.txt'), 'w') as f:
    f.write(f"repository_commit: {commit}\n")
    f.write(f"zipped_at_utc: {datetime.datetime.utcnow().isoformat()}Z\n")
    f.write(f"device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}\n")

with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    for folder in (out_dir, out_dir + '_smoke'):
        if os.path.exists(folder):
            for root, _, files in os.walk(folder):
                for fname in files:
                    if fname.startswith('temp_'):
                        continue
                    full = os.path.join(root, fname)
                    zf.write(full, os.path.join(os.path.basename(folder), os.path.relpath(full, folder)))
    zf.write('Handover Scheduling/colab/run_signal_selection_validation.py', 'code/run_signal_selection_validation.py')

print(f"Created {zip_name} ({os.path.getsize(zip_name) / 1e6:.2f} MB)")
with zipfile.ZipFile(zip_name) as zf:
    for n in zf.namelist():
        print("  ", n)

try:
    from google.colab import files
    files.download(zip_name)
except Exception:
    print(f"Download manually from the Files panel: /content/Shonget-DMOSpeech2/{zip_name}")